# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

> You do not need to run the following cells if you are running this notebook locally. 

In [10]:
pip install -qU langchain langchain-openai langchain-cohere rank_bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-experimental 0.0.65 requires langchain-community<0.3.0,>=0.2.16, but you have langchain-community 0.3.24 which is incompatible.
langchain-experimental 0.0.65 requires langchain-core<0.3.0,>=0.2.38, but you have langchain-core 0.3.60 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


We're also going to be leveraging [Qdrant's](https://qdrant.tech/documentation/frameworks/langchain/) (pronounced "Quadrant") VectorDB in "memory" mode (so we can leverage it locally in our colab environment).

In [11]:
pip install -qU qdrant-client

Note: you may need to restart the kernel to use updated packages.


We'll also provide our OpenAI key, as well as our Cohere API key.

In [12]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [13]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using some reviews from the 4 movies in the John Wick franchise today to explore the different retrieval strategies.

These were obtained from IMDB, and are available in the [AIM Data Repository](https://github.com/AI-Maker-Space/DataRepository).

### Data Collection

We can simply `wget` these from GitHub.

You could use any review data you wanted in this step - just be careful to make sure your metadata is aligned with your choice.

In [14]:
!curl -o john_wick_1.csv https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw1.csv
!curl -o john_wick_2.csv https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw2.csv
!curl -o john_wick_3.csv https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw3.csv
!curl -o john_wick_4.csv https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw4.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     019628  100 19628    0     0   133k      0 --:--:-- --:--:-- --:--:--  134k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 14747  100 14747    0     0   102k      0 --:--:-- --:--:-- --:--:--  102k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13  0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0888  100 13888    0     0   100k      0 --:--:-- --:--:-- --:--:--  100k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent 

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

- Self-Query: Wants as much metadata as we can provide
- Time-weighted: Wants temporal data

> NOTE: While we're creating a temporal relationship based on when these movies came out for illustrative purposes, it needs to be clear that the "time-weighting" in the Time-weighted Retriever is based on when the document was *accessed* last - not when it was created.

In [15]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

documents = []

for i in range(1, 5):
  loader = CSVLoader(
      file_path=f"john_wick_{i}.csv",
      metadata_columns=["Review_Date", "Review_Title", "Review_Url", "Author", "Rating"]
  )

  movie_docs = loader.load()
  for doc in movie_docs:

    # Add the "Movie Title" (John Wick 1, 2, ...)
    doc.metadata["Movie_Title"] = f"John Wick {i}"

    # convert "Rating" to an `int`, if no rating is provided - assume 0 rating
    doc.metadata["Rating"] = int(doc.metadata["Rating"]) if doc.metadata["Rating"] else 0

    # newer movies have a more recent "last_accessed_at"
    doc.metadata["last_accessed_at"] = datetime.now() - timedelta(days=4-i)

  documents.extend(movie_docs)

Let's look at an example document to see if everything worked as expected!

In [16]:
documents[0]

Document(metadata={'source': 'john_wick_1.csv', 'row': 0, 'Review_Date': '6 May 2015', 'Review_Title': ' Kinetic, concise, and stylish; John Wick kicks ass.\n', 'Review_Url': '/review/rw3233896/?ref_=tt_urv', 'Author': 'lnvicta', 'Rating': 8, 'Movie_Title': 'John Wick 1', 'last_accessed_at': datetime.datetime(2025, 5, 17, 17, 31, 10, 693123)}, page_content=": 0\nReview: The best way I can describe John Wick is to picture Taken but instead of Liam Neeson it's Keanu Reeves and instead of his daughter it's his dog. That's essentially the plot of the movie. John Wick (Reeves) is out to seek revenge on the people who took something he loved from him. It's a beautifully simple premise for an action movie - when action movies get convoluted, they get bad i.e. A Good Day to Die Hard. John Wick gives the viewers what they want: Awesome action, stylish stunts, kinetic chaos, and a relatable hero to tie it all together. John Wick succeeds in its simplicity.")

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "JohnWick".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [17]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    documents,
    embeddings,
    location=":memory:",
    collection_name="JohnWick"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [18]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [19]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [20]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")


### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [21]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [22]:
naive_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content

"Based on the reviews provided, people generally liked John Wick. Several reviews gave it high ratings (such as 9 and 10 out of 10) and described it as stylish, fun, and an exciting action film. Many critics praised its action sequences, performances (especially Keanu Reeves'), and its stylish direction. While there were some mixed opinions and a few lower ratings, the overall sentiment indicates that most people appreciated and enjoyed the film."

In [23]:
naive_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content

'Yes, there are reviews with a rating of 10. One of them is from the review by ymuuseda for John Wick 3. \n\nThe URL for this review is: https://yourdomain.com/review/rw4854296/?ref_=tt_urv'

In [24]:
naive_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content

"In John Wick, the main character is a retired hitman named John Wick, played by Keanu Reeves. The story begins with the death of his wife, which leaves him feeling lost. His life takes a turn when a young Russian punk attempts to buy his cherished car, and later, with the help of his assailants, kills his dog—an act that deeply disturbs him since the dog was likely a symbol of his last connection to his wife. Discovering that his dog is dead and his car stolen, Wick, a legendary assassin, is forced back into violence to seek vengeance. \n\nThroughout the series, John Wick battles various criminal factions, including Russian gangsters and members of the assassin's community, as he tries to recover what was taken from him and deal with the consequences of his past actions. The films feature a mix of stylish action sequences, elaborate fight scenes, and a deep dive into a secret underworld of assassins and criminal organizations."

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [25]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents)

We'll construct the same chain - only changing the retriever.

In [26]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [27]:
bm25_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content

"People's opinions on John Wick vary depending on the movie and reviewer. For example, reviews of the first film are highly positive, praising its action, style, and simplicity, with ratings like 8 and 10 out of 10. However, some later films receive negative reviews; for instance, the third movie was criticized as dull, stereotypical, and overly violent, with a rating of only 1 out of 10. The fourth movie received a mixed review, with the reviewer describing it as almost three hours of what they considered filler and calling it the weakest in the series. \n\nOverall, it appears that many people liked John Wick, especially the first film, but opinions on subsequent installments are more divided, with some viewers disliking them."

In [28]:
bm25_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content

'Based on the provided reviews, there are no reviews with a rating of 10.'

In [29]:
bm25_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content

'In the John Wick film series, the story centers around John Wick, a former assassin who is drawn back into the violent underworld he left behind. The movies depict his intense battles against various enemies, including assassins and crime organizations, as he seeks vengeance and survival. The series is known for its beautifully choreographed action scenes, emotional depth, and a fictional universe filled with rules and codes governing the assassin community.'

It's not clear that this is better or worse - but the `I don't know` isn't great!

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [30]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-english-v3.0")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [31]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [32]:
contextual_compression_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content

'Based on the reviews in the provided context, people generally liked John Wick. The reviews are highly positive, praising the film\'s action sequences, style, and Keanu Reeves\' performance. For example, one reviewer rated it a 9 out of 10, calling it "the coolest action film you\'ll see all year," and another gave it a perfect 10, describing it as "something special." However, there is a less favorable review for the third installment, which rated it a 5 out of 10 and felt the "magic is gone." Overall, the majority of the reviews in the context suggest that people generally enjoyed the film.'

In [33]:
contextual_compression_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content

"Yes, there are reviews with a rating of 10. Here are the URLs to those reviews:\n\n1. [Review from john_wick_3.csv](https://example.com/review/rw4854296/?ref_=tt_urv)\n\n(Note: The actual review URL is provided in the context as '/review/rw4854296/?ref_=tt_urv', but since I cannot browse the web, I have presented it as it appears. If you need the full URL, you may need to append the base website address accordingly.)"

In [34]:
contextual_compression_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content

"In the John Wick series, John Wick (played by Keanu Reeves) is a retired hitman who is drawn back into a violent underworld. In the first film, he seeks revenge after gangsters kill his dog and steal his car, leading him into a relentless quest for vengeance. In the second installment, after resolving issues with the Russian mafia, John is approached by a mobster, Santino D'Antonio, who enlists his help to kill his sister in Rome so he can sit on the High Table of criminal organizations. John completes the mission, but Santino then puts a bounty on his head, turning him into a target for numerous professional killers. The series includes intense action, revenge plots, and a world of organized crime with strict rules."

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [35]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [36]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [37]:
multi_query_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content

'Based on the reviews in the provided context, people generally liked John Wick. Several reviews give high ratings, praising the film\'s action, style, and entertainment value. For example, one review rated it a 9 and called it "the coolest action film you\'ll see all year," while another rated it a 10 and described it as "smoothest action film to come around in a long time." Many reviewers highlight its fun, brutal action sequences, and engaging world-building, indicating that it was well-received by audiences. \n\nHowever, there are some mixed reviews with lower ratings—such as a 4 or 5—who found the film\'s over-the-top violence or story lacking, but these are less common. Overall, the majority of the feedback suggests that people generally liked John Wick.'

In [38]:
multi_query_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content

'Yes, there is a review with a rating of 10. The URL to that review is: /review/rw4854296/?ref_=tt_urv'

In [39]:
multi_query_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content

"In the John Wick series, the story centers around John Wick, a retired, highly skilled assassin who is drawn back into the violent world of killing after personal tragedies and events unfold. The first film depicts how Wick comes out of retirement to seek revenge after gangsters kill his dog and steal his car, both of which hold deep emotional significance to him. As he re-enters this dangerous underworld, he faces numerous enemies and a bounty on his head, leading to many action-packed confrontations.\n\nSubsequent films expand on this universe, exploring Wick’s complex history, the criminal underworld's rules, and the consequences of his actions. The series is known for its stylish, choreographed action sequences and its portrayal of a gritty, shadowy world of assassins, where every action has repercussions.\n\nOverall, John Wick's story is a blend of vengeance, survival, and the exploration of a dark, deadly underworld, with Keanu Reeves portraying the iconic and relentless assassi

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [40]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = documents
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [41]:
client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = Qdrant(
    collection_name="full_documents", embeddings=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

/var/folders/1s/7jt3wwjx19g99p8p8_kv3_sw0000gp/T/ipykernel_58754/3574430551.py:8: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-qdrant package and should be used instead. To use it run `pip install -U :class:`~langchain-qdrant` and import as `from :class:`~langchain_qdrant import Qdrant``.
  parent_document_vectorstore = Qdrant(


Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [42]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [43]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [44]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [45]:
parent_document_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content

'Based on the reviews provided, people\'s opinions on John Wick vary. Some reviews are very positive, praising the series and the movies, with one rating it highly and calling the first film "highly recommend." However, there is at least one negative review calling "John Wick 4" horrible and criticizing its plot and fight scenes. \n\nOverall, while many seem to enjoy the series, there are also some negative opinions. Therefore, it can be said that people\'s reactions to John Wick are mixed, with many liking it, but some not.'

In [46]:
parent_document_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content

'Yes, there is at least one review with a rating of 10. The URL to that review is: /review/rw4854296/?ref_=tt_urv'

In [47]:
parent_document_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content

"In the John Wick movies, John Wick is a retired assassin who comes out of retirement to seek vengeance and resolve various conflicts. In the first film, he is motivated by the killing of his dog and theft of his car, which leads him to go after gangsters and criminals to settle personal scores. The story involves intense action, revenge, and uncovering his own hidden identity.\n\nIn the second movie, the plot continues with John Wick being pulled back into the world of assassins when an old associate demands his help to settle a debt. This results in him traveling to Italy, Canada, and Manhattan to eliminate numerous enemies and face new threats.\n\nOverall, the series is characterized by high-octane action, revenge, and the exploration of John Wick's lethal skills and complex world."

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [48]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [49]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [50]:
ensemble_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content

"Based on the reviews in the provided context, people generally liked John Wick. Many reviews praise its stylish action sequences, Keanu Reeves' performance, and its status as a fun, brutal, and well-choreographed action film. Several reviews give high ratings (such as 8, 9, and 10 out of 10), indicating positive reception. However, there are some negative reviews as well, particularly for the later installments, with ratings around 1 to 4, criticizing aspects like plot, violence levels, or over-the-top action. Overall, the majority of reviews seem to be favorable, suggesting that people generally enjoyed the series."

In [51]:
ensemble_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content

'Yes, there are reviews with a rating of 10. Here are the URLs to those reviews:\n\n1. [https://yourdomain.com/review/rw4854296/?ref_=tt_urv](https://yourdomain.com/review/rw4854296/?ref_=tt_urv) - Review titled "A Masterpiece & Brilliant Sequel" for John Wick 3, dated 15 May 2019.\n\n2. [https://yourdomain.com/review/rw8944843/?ref_=tt_urv](https://yourdomain.com/review/rw8944843/?ref_=tt_urv) - Review titled "How Can Anyone Choose to Watch Marvel Over This?" for John Wick 4, dated 22 March 2023.\n\nPlease note that the URLs are based on the Review_Url indicated in the data, formatted as full links for your convenience.'

In [52]:
ensemble_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content

"In the John Wick series, the story revolves around a retired assassin named John Wick, played by Keanu Reeves. The original film begins with Wick mourning the death of his wife and trying to find peace. His life is shattered when a gang of thugs, including a young punk, break into his house, kill his dog—a final gift from his wife—and steal his car. This brutal assault awakens Wick's deadly skills, and he seeks revenge against those who wronged him, unleashing a violent and meticulously choreographed rampage against his enemies.\n\nAs the series progresses, Wick is drawn back into the world of espionage and criminal underworld conflicts. In subsequent films, he is tasked with impossible missions, such as helping to take over the Assassin’s Guild or killing a crime family member to settle old debts. Throughout the series, Wick faces numerous professional killers and must navigate complex rules of the underworld, often with a bounty on his head. The films are renowned for their stylish 

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

> NOTE: You do not need to run this cell if you're running this locally

In [53]:
pip install -qU langchain_experimental

Note: you may need to restart the kernel to use updated packages.


We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [54]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [55]:
semantic_documents = semantic_chunker.split_documents(documents)

Let's create a new vector store.

In [56]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="JohnWickSemantic"
)

We'll use naive retrieval for this example.

In [57]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [58]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [59]:
semantic_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content

'Yes, people generally liked John Wick. The reviews included in the context are mostly very positive, with ratings of 8, 9, and 10 out of 10, and commentary highlighting its style, action sequences, and entertainment value.'

In [60]:
semantic_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content

'Yes, there is a review with a rating of 10. The URL to that review is /review/rw4854296/?ref_=tt_urv.'

In [61]:
semantic_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content

"In the John Wick movies, the main story revolves around a retired assassin named John Wick (played by Keanu Reeves) who is drawn back into a lethal world of violence and revenge. The first film's plot begins when a young Russian punk, who notices Wick's valuable car, tries to buy it from him. Wick declines, but soon the punk's associates surprise him at his home, beating him up, killing his beloved dog, and stealing his car. Unbeknownst to them, Wick is a former legendary hit-man. This brutal assault triggers Wick's pursuit of revenge against those who wronged him, leading to a violent rampage against the Russian mobsters and others involved, as he seeks vengeance and justice for the loss of his dog and personal peace. The series explores themes of vengeance, the consequences of violence, and Wick's relentless fight against those who threaten him."

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [114]:
import os

os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Enter your LangSmith API Key:")
os.environ["LANGCHAIN_PROJECT"] = "ragas-evaluation"


Generate a Synthetic Golden Dataset

In [102]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from datasets import Dataset

# Your trimmed subset
subset_docs = documents[:5]

# LangChain LLM — this is traceable by LangSmith
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Prompt template
prompt = ChatPromptTemplate.from_template("""
Given the following movie review content, generate one useful question a user might ask about it,
and provide an accurate answer based on the review.

Review Content:
{content}

Format your response like:
Question: ...
Answer: ...
""")

# Chain = Prompt + LLM
qa_chain = prompt | llm

# Collect generated QA
questions, answers = [], []

for doc in subset_docs:
    content = doc.page_content[:2000]
    result = qa_chain.invoke({"content": content}).content.strip()
    if "Answer:" in result:
        q, a = result.split("Answer:", 1)
        questions.append(q.replace("Question:", "").strip())
        answers.append(a.strip())


In [103]:
# Use original doc as the retrieval context
contexts = [[doc.page_content] for doc in subset_docs[:len(questions)]]

golden_dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts
})

# Preview
golden_dataset.to_pandas().head()


,question,answer,contexts
0,How does John Wick differentiate itself from o...,John Wick sets itself apart by having a simple...,[: 0\nReview: The best way I can describe John...
1,How does this movie differ from other contempo...,This movie stands out from other contemporary ...,[: 1\nReview: It looks as if the filmmakers re...
2,How does the reviewer feel about the previous ...,The reviewer mentions that the three previous ...,[: 2\nReview: With the fourth installment scor...
3,What sets John Wick apart from other action mo...,What sets John Wick apart from other action mo...,[: 3\nReview: John wick has a very simple reve...
4,Is John Wick: Chapter 2 worth watching for fan...,"Based on the review, John Wick: Chapter 2 is p...",[: 4\nReview: Though he no longer has a taste ...


Evaluate Retrieval Performance

1. metrics

In [104]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

Evaluation Function Template

In [116]:
from langsmith import traceable

def evaluate_retriever(chain, dataset, label):
    def generate(example):
        # manually break apart so we trace only the LLM step
        retrieved = chain.invoke({"question": example["question"]})["context"]
        
        @traceable(name=label)
        def call_llm(context):
            return chat_model.invoke(f"""
            You are a helpful assistant. Use the context to answer the question.

            Question: {example['question']}
            Context: {context}
            """)
        
        return call_llm(retrieved).content

    dataset_with_preds = dataset.map(lambda x: {"response": generate(x)})

    results = evaluate(
        dataset=dataset_with_preds,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision,
            context_recall,
        ],
    )

    return results.to_pandas()


Naive Retriever

In [118]:
naive_results = evaluate_retriever(naive_retrieval_chain, golden_dataset, "Naive Retriever")
print("Naive Retriever Results:")
print(naive_results)

Evaluating: 100%|██████████| 20/20 [00:33<00:00,  1.68s/it]


Naive Retriever Results:
                                          user_input  \
0  How does John Wick differentiate itself from o...   
1  How does this movie differ from other contempo...   
2  How does the reviewer feel about the previous ...   
3  What sets John Wick apart from other action mo...   
4  Is John Wick: Chapter 2 worth watching for fan...   

                                  retrieved_contexts  \
0  [: 0\nReview: The best way I can describe John...   
1  [: 1\nReview: It looks as if the filmmakers re...   
2  [: 2\nReview: With the fourth installment scor...   
3  [: 3\nReview: John wick has a very simple reve...   
4  [: 4\nReview: Though he no longer has a taste ...   

                                            response  \
0  John Wick differentiates itself from other act...   
1  This movie, particularly the John Wick series,...   
2  The reviewer has mixed feelings about the prev...   
3  What sets John Wick apart from other action mo...   
4  Yes, John Wick: Ch

BM25 Retriever

In [119]:
bm25_results = evaluate_retriever(bm25_retrieval_chain, golden_dataset, "BM25 Retriever")
print("\n🔹 BM25 Retriever Results:")
print(bm25_results)

Evaluating: 100%|██████████| 20/20 [00:24<00:00,  1.24s/it]



🔹 BM25 Retriever Results:
                                          user_input  \
0  How does John Wick differentiate itself from o...   
1  How does this movie differ from other contempo...   
2  How does the reviewer feel about the previous ...   
3  What sets John Wick apart from other action mo...   
4  Is John Wick: Chapter 2 worth watching for fan...   

                                  retrieved_contexts  \
0  [: 0\nReview: The best way I can describe John...   
1  [: 1\nReview: It looks as if the filmmakers re...   
2  [: 2\nReview: With the fourth installment scor...   
3  [: 3\nReview: John wick has a very simple reve...   
4  [: 4\nReview: Though he no longer has a taste ...   

                                            response  \
0  John Wick differentiates itself from other act...   
1  This movie, particularly the John Wick series,...   
2  The reviewer does not feel particularly positi...   
3  John Wick is set apart from other action movie...   
4  Yes, John Wick: 

Contextual Compression (Rerank with Cohere)

In [120]:
compression_results = evaluate_retriever(contextual_compression_retrieval_chain, golden_dataset, "Contextual Compression")
print("\n🔹 Contextual Compression Retriever Results:")
print(compression_results)

Evaluating: 100%|██████████| 20/20 [00:21<00:00,  1.07s/it]



🔹 Contextual Compression Retriever Results:
                                          user_input  \
0  How does John Wick differentiate itself from o...   
1  How does this movie differ from other contempo...   
2  How does the reviewer feel about the previous ...   
3  What sets John Wick apart from other action mo...   
4  Is John Wick: Chapter 2 worth watching for fan...   

                                  retrieved_contexts  \
0  [: 0\nReview: The best way I can describe John...   
1  [: 1\nReview: It looks as if the filmmakers re...   
2  [: 2\nReview: With the fourth installment scor...   
3  [: 3\nReview: John wick has a very simple reve...   
4  [: 4\nReview: Though he no longer has a taste ...   

                                            response  \
0  John Wick differentiates itself from other act...   
1  This movie, John Wick, differs from other cont...   
2  The reviewer feels positively about the previo...   
3  John Wick is set apart from other action movie...   
4

Multi-Query Retriever

In [121]:
multi_query_results = evaluate_retriever(multi_query_retrieval_chain, golden_dataset, "Multi-Query Retriever")
print("\n🔹 Multi-Query Retriever Results:")
print(multi_query_results)

Evaluating: 100%|██████████| 20/20 [00:28<00:00,  1.40s/it]



🔹 Multi-Query Retriever Results:
                                          user_input  \
0  How does John Wick differentiate itself from o...   
1  How does this movie differ from other contempo...   
2  How does the reviewer feel about the previous ...   
3  What sets John Wick apart from other action mo...   
4  Is John Wick: Chapter 2 worth watching for fan...   

                                  retrieved_contexts  \
0  [: 0\nReview: The best way I can describe John...   
1  [: 1\nReview: It looks as if the filmmakers re...   
2  [: 2\nReview: With the fourth installment scor...   
3  [: 3\nReview: John wick has a very simple reve...   
4  [: 4\nReview: Though he no longer has a taste ...   

                                            response  \
0  John Wick differentiates itself from other act...   
1  This movie, particularly the "John Wick" serie...   
2  The reviewer has mixed feelings about the prev...   
3  John Wick stands out from other action movies ...   
4  Yes, John

Parent Document Retriever

In [122]:
parent_doc_results = evaluate_retriever(parent_document_retrieval_chain, golden_dataset, "Parent Document Retriever")
print("\n🔹 Parent Document Retriever Results:")
print(parent_doc_results)

Evaluating: 100%|██████████| 20/20 [00:17<00:00,  1.15it/s]



🔹 Parent Document Retriever Results:
                                          user_input  \
0  How does John Wick differentiate itself from o...   
1  How does this movie differ from other contempo...   
2  How does the reviewer feel about the previous ...   
3  What sets John Wick apart from other action mo...   
4  Is John Wick: Chapter 2 worth watching for fan...   

                                  retrieved_contexts  \
0  [: 0\nReview: The best way I can describe John...   
1  [: 1\nReview: It looks as if the filmmakers re...   
2  [: 2\nReview: With the fourth installment scor...   
3  [: 3\nReview: John wick has a very simple reve...   
4  [: 4\nReview: Though he no longer has a taste ...   

                                            response  \
0  John Wick differentiates itself from other act...   
1  This movie, particularly "John Wick," differs ...   
2  The reviewer feels positively about the previo...   
3  John Wick is set apart from other action movie...   
4  Based

Ensemble Retriever

In [123]:
ensemble_results = evaluate_retriever(ensemble_retrieval_chain, golden_dataset, "Ensemble Retriever")
print("\n🔹 Ensemble Retriever Results:")
print(ensemble_results)

Evaluating: 100%|██████████| 20/20 [00:24<00:00,  1.21s/it]



🔹 Ensemble Retriever Results:
                                          user_input  \
0  How does John Wick differentiate itself from o...   
1  How does this movie differ from other contempo...   
2  How does the reviewer feel about the previous ...   
3  What sets John Wick apart from other action mo...   
4  Is John Wick: Chapter 2 worth watching for fan...   

                                  retrieved_contexts  \
0  [: 0\nReview: The best way I can describe John...   
1  [: 1\nReview: It looks as if the filmmakers re...   
2  [: 2\nReview: With the fourth installment scor...   
3  [: 3\nReview: John wick has a very simple reve...   
4  [: 4\nReview: Though he no longer has a taste ...   

                                            response  \
0  John Wick differentiates itself from other act...   
1  This movie differs from other contemporary fil...   
2  The reviewer generally feels positively about ...   
3  What sets John Wick apart from other action mo...   
4  Yes, John Wi

In [124]:
import pandas as pd

def compile_retriever_results(
    naive, bm25, compression, multi_query, parent_doc, ensemble
) -> pd.DataFrame:
    return pd.concat([
        naive.assign(retriever="Naive"),
        bm25.assign(retriever="BM25"),
        compression.assign(retriever="Compression"),
        multi_query.assign(retriever="Multi-Query"),
        parent_doc.assign(retriever="Parent Document"),
        ensemble.assign(retriever="Ensemble"),
    ])[
        ["retriever", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    ].reset_index(drop=True)


In [125]:
final_results = compile_retriever_results(
    naive_results,
    bm25_results,
    compression_results,
    multi_query_results,
    parent_doc_results,
    ensemble_results
)

final_results


,retriever,faithfulness,answer_relevancy,context_precision,context_recall
0,Naive,0.350000,0.985216,1.0,1.0
1,Naive,0.111111,0.924765,1.0,1.0
2,Naive,0.047619,0.000000,1.0,1.0
3,Naive,0.500000,0.986704,1.0,1.0
4,Naive,0.000000,1.000000,1.0,0.5
5,BM25,0.000000,0.985216,1.0,1.0
6,BM25,0.300000,0.924765,1.0,1.0
7,BM25,0.571429,0.000000,1.0,1.0
8,BM25,0.125000,0.986704,1.0,1.0
9,BM25,0.000000,1.000000,1.0,0.5


In [126]:
def compute_retriever_averages(df):
    """
    Takes a combined RAGAS results DataFrame and returns average metrics per retriever.
    
    Expected columns: ['retriever', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
    """
    grouped = df.groupby("retriever").agg({
        "faithfulness": "mean",
        "answer_relevancy": "mean",
        "context_precision": "mean",
        "context_recall": "mean"
    }).round(4).reset_index()
    
    return grouped.sort_values(by="faithfulness", ascending=False)


In [127]:
average_results = compute_retriever_averages(final_results)
average_results


,retriever,faithfulness,answer_relevancy,context_precision,context_recall
2,Ensemble,0.3376,0.9893,1.0,0.9
1,Compression,0.2523,0.9636,1.0,0.9
5,Parent Document,0.2239,0.9606,1.0,0.9
4,Naive,0.2017,0.7793,1.0,0.9
0,BM25,0.1993,0.7793,1.0,0.9
3,Multi-Query,0.0812,0.7785,1.0,0.9


After testing all six retrievers using RAGAS, the Ensemble Retriever came out on top. It had the best overall performance, especially in faithfulness and answer relevancy, while maintaining perfect context precision and solid recall.

That makes sense since it combines the strengths of multiple strategies, so it pulls in more useful context. But it’s also likely the most expensive and slowest, since it runs several retrievers in parallel and uses more tokens.

If performance is the only priority, Ensemble is the best option. But if I had to balance cost or latency, I’d go with Contextual Compression or Parent Document Retriever. They still scored well across all metrics and would be more efficient for real use.

LangSmith tracing didn’t work due to a local issue, but we estimated cost and latency based on how each retriever works.